In [4]:
import json
import os
from datetime import datetime
import numpy as np
# Used to securely store your API key
from google.colab import userdata

# Install faiss-cpu if not already installed
!pip install faiss-cpu > /dev/null 2>&1

# If using OpenAI
from openai import OpenAI

# -------------------------------
# CONFIG
# -------------------------------
LOG_FILE = "rag_logs.jsonl"   # JSONL format (one JSON per line)
TOP_K = 3
SIMILARITY_THRESHOLD = 0.5

# Load API key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

# -------------------------------
# LOGGING FUNCTION
# -------------------------------
def log_result(data):
    data["timestamp"] = datetime.now().isoformat()

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False) + "\n")


# -------------------------------
# EMBEDDING FUNCTION
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding)


# -------------------------------
# RETRIEVAL FUNCTION (FAISS)
# -------------------------------
def retrieve(query, index, documents, k=TOP_K):
    query_embedding = get_embedding(query)

    distances, indices = index.search(
        np.array([query_embedding]).astype("float32"), k
    )

    retrieved_chunks = []
    similarity_scores = []

    for i, idx in enumerate(indices[0]):
        if idx == -1:
            continue

        score = float(distances[0][i])

        retrieved_chunks.append({
            "doc_id": int(idx),
            "content": documents[idx]
        })
        similarity_scores.append(score)

    return retrieved_chunks, similarity_scores


# -------------------------------
# GENERATION FUNCTION
# -------------------------------
def generate_answer(query, retrieved_chunks):
    context = "\n\n".join([chunk["content"] for chunk in retrieved_chunks])

    prompt = f"""
You are a helpful AI assistant.
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer only from context. If not found, say 'Not found'."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# -------------------------------
# MAIN RAG PIPELINE
# -------------------------------
def rag_pipeline(query, index, documents):
    retrieved_chunks, similarity_scores = retrieve(query, index, documents)

    answer = generate_answer(query, retrieved_chunks)

    # Log everything
    log_data = {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "similarity_scores": similarity_scores,
        "final_answer": answer
    }

    log_result(log_data)

    return answer


# -------------------------------
# FAILURE CLASSIFICATION (OPTIONAL)
# -------------------------------
def classify_failure(similarity_scores, answer):
    if len(similarity_scores) == 0:
        return "retrieval_failure"

    if max(similarity_scores) < SIMILARITY_THRESHOLD:
        return "vague_context"

    if "Not found" in answer:
        return "answer_context_mismatch"

    return "success"


# -------------------------------
# RUN TEST SUITE
# -------------------------------
def run_test_suite(queries, index, documents):
    results = []

    for q in queries:
        print(f"\n🔍 Query: {q}")

        retrieved_chunks, similarity_scores = retrieve(q, index, documents)
        answer = generate_answer(q, retrieved_chunks)

        failure_type = classify_failure(similarity_scores, answer)

        result = {
            "query": q,
            "retrieved_chunks": retrieved_chunks,
            "similarity_scores": similarity_scores,
            "final_answer": answer,
            "failure_type": failure_type
        }

        log_result(result)
        results.append(result)

        print(f"Answer: {answer}")
        print(f"Failure Type: {failure_type}")

    return results


# -------------------------------
# SCORECARD
# -------------------------------
def score_results(results):
    retrieval_scores = []
    answer_scores = []

    for r in results:
        # Simple heuristic scoring
        retrieval_score = min(5, len(r["retrieved_chunks"]))
        answer_score = 5 if "Not found" not in r["final_answer"] else 2

        retrieval_scores.append(retrieval_score)
        answer_scores.append(answer_score)

    avg_retrieval = sum(retrieval_scores) / len(retrieval_scores)
    avg_answer = sum(answer_scores) / len(answer_scores)

    print("\n📊 SCORECARD")
    print(f"Average Retrieval Score: {avg_retrieval:.2f}")
    print(f"Average Answer Score: {avg_answer:.2f}")


# -------------------------------
# EXAMPLE USAGE
# -------------------------------
if __name__ == "__main__":
    # Dummy example (replace with your FAISS index + docs)
    documents = [
        "RAG stands for Retrieval-Augmented Generation.",
        "FAISS is a similarity search library by Facebook.",
        "Embeddings convert text into vectors."
    ]

    import faiss

    # Build FAISS index (example)
    embeddings = np.array([get_embedding(doc) for doc in documents]).astype("float32")
    dim = embeddings.shape[1]

    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    # Test queries
    queries = [
        "What is RAG?",
        "What is FAISS?",
        "Explain embeddings",
        "What is machine learning?"
    ]

    results = run_test_suite(queries, index, documents)

    score_results(results)

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.